In [ ]:
import functools
import warnings

import botocore
import boto3
from iterpop import iterpop as ip
from matplotlib.markers import MarkerStyle
from matplotlib.ticker import MultipleLocator
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import pandas as pd
from pandas.util import hash_pandas_object
from scipy import stats as scipy_stats
import seaborn as sns
from teeplot import teeplot as tp
from tqdm import tqdm

from dishpylib.pyhelpers import fit_control_t_distns

warnings.filterwarnings("ignore")


In [ ]:
from dishpylib.pyhelpers import print_runtime


In [ ]:
print_runtime()


In [ ]:
teeplot_subdir = "2026-06-22-eco-gwas"


In [ ]:
@functools.lru_cache
def get_control_t_distns( bucket, prefix, suffix, endeavor, stint ):

    s3_handle = boto3.resource(
        's3',
        region_name="us-east-2",
        config=botocore.config.Config(
            signature_version=botocore.UNSIGNED,
        ),
    )
    bucket_handle = s3_handle.Bucket(bucket)

    control_competitions, = bucket_handle.objects.filter(
        Prefix=f'endeavor={endeavor}/{prefix}control-competitions-highestroot{suffix}/stage={4 + bool(prefix)}+what=collated/stint={stint}/',
    )

    control_df = pd.read_csv(
        f's3://{bucket}/{control_competitions.key}',
    )

    return fit_control_t_distns(control_df[
        control_df["Root ID"].isin([0, 1])
    ].copy())


In [ ]:
def preprocess_competition_fitnesses(competitions_df, control_fits_df):
    print(len(competitions_df), "competitions to preprocess")
    print(len(control_fits_df), "control fits available")
    # preprocess data
    @functools.lru_cache
    def h0_fit(series):
        return ip.popsingleton(
            control_fits_df[control_fits_df["Series"] == series].to_dict(
                orient="records",
            )
        )

    competitions_df["p"] = competitions_df.apply(
        lambda row: scipy_stats.t.cdf(
            row["Fitness Differential"],
            h0_fit(row["genome series"])["Fit Degrees of Freedom"],
            loc=h0_fit(row["genome series"])["Fit Loc"],
            scale=h0_fit(row["genome series"])["Fit Scale"],
        ),
        axis=1,
    )
    competitions_df["Is Less Fit"] = competitions_df["p"] < 1.0 / 40
    competitions_df["Is More Fit"] = competitions_df["p"] > (1.0 -  1.0 / 40)
    competitions_df["Is Neutral"] = ~(
        competitions_df["Is Less Fit"] | competitions_df["Is More Fit"]
    )
    competitions_df["Relative Fitness"] = competitions_df.apply(
        lambda row: (
            "Significantly Advantageous"
            if row["Is More Fit"]
            else (
                "Significantly Deleterious" if row["Is Less Fit"] else "Neutral"
            )
        ),
        axis=1,
    )

    return competitions_df


# get data


In [ ]:
bucket = "prq49"
dfs = []
for suffix in [
    # "cryptic-",
    "-backgroundbb",
    "-focalbb",

]:
    prefix = ""
    if "step" in bucket:
        step = int(bucket.split("-step")[-1]) + 1
    else:
        step = 0
    if "restint" in bucket:
        kind = bucket.split("-")[3]
    else:
        kind = None
    s3_handle = boto3.resource(
        's3',
        region_name="us-east-2",
        config=botocore.config.Config(
            signature_version=botocore.UNSIGNED,
        ),
    )
    bucket_handle = s3_handle.Bucket(bucket)

    for stint in tqdm([100]):
        try:
            series_profiles, = bucket_handle.objects.filter(
                Prefix=f'endeavor=16/{prefix}variant-competitions-highestroot{suffix}/stage=4+what=collated/stint={stint}/',
            )
            import warnings
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
            control_fits_df = get_control_t_distns(bucket, prefix, suffix, 16, stint)
            df = pd.read_csv(
                f's3://{bucket}/{series_profiles.key}',
                compression='xz',
            )
            # df = df[df["Competition Series"] == 16005]
            # df = df.groupby([
            #     "genome variation"
            # ]).mean(numeric_only=True).reset_index()
            df["Stint"] = stint
            df["Series"] = df["Competition Series"]
            dfdigest = "{:x}".format( hash_pandas_object( df ).sum() )
            df = preprocess_competition_fitnesses(df, control_fits_df)
            assert "Series" in df.columns, df.columns

            df = df.copy()
            df["bucket"] = bucket
            df["variant"] = {"cryptic-": "skeleton", "": "wildtype"}[prefix]
            df["how"] = suffix.strip("-") if suffix else "none"
            dfs.append(df)
        except Exception as e:
            print(e)
            print(f"Skipping {bucket=}, {stint=}, {prefix=}")

print(len(dfs), "dataframes loaded")


In [ ]:
df = pd.concat(dfs)


In [ ]:
pd.options.display.max_columns = None


In [ ]:
dfx = df[df["Root ID"] == 1]
dfx["site"] = dfx["genome variation"].str.extract(r'i(\d+)%')


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

dfx["site"] = dfx["site"].astype(int)  # Ensure site is numeric for proper sorting

# 1. Isolate and filter the sites (same as before)
valid_sites = dfx[dfx["Prevalence"] < 0.2]["site"].unique()
df_filtered = dfx[dfx["site"].isin(valid_sites)]

# 2. Draw the scatterplot
ax = sns.scatterplot(
    data=df_filtered,
    x="site",
    y="Prevalence",
    style="how",
    hue="how",
    s=50,      # Slightly larger dots look better on connected plots
    zorder=2,   # Forces the dots to be drawn ON TOP of the lines
    alpha=0.9,
    linewidth=1.5,
)

sns.scatterplot(
    data=dfx,
    x="site",
    y="Prevalence",
    style="how",
    hue="how",
    s=7,      # Slightly larger dots look better on connected plots
    alpha=0.3,  # Make the background points more transparent
    zorder=-1,  # Forces the dots to be drawn BEHIND the lines
    legend=False,  # Avoid duplicate legends
    ax=ax,
)

# 3. Find the min and max for each site to get our line endpoints
ranges = df_filtered.groupby("site")["Prevalence"].agg(["min", "max"])

# 4. Draw the connecting vertical lines
ax.vlines(
    x=ranges.index,
    ymin=ranges["min"],
    ymax=ranges["max"],
    color="k",
    linestyle=":", # Optional: makes the connection a dashed line
    linewidth=1,   # Optional: makes the line thinner
    alpha=0.9,      # Optional: softens the line color
    zorder=1        # Forces the lines to be drawn BEHIND the dots
)

# 5. Position the legend
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
ax.axhline(0.25, color='gray', linestyle='--', zorder=-1, alpha=0.5)  # Optional: adds a horizontal reference line at y=0.25


In [ ]:
bucket = "prq49"
dfs = []
for suffix in [
    # "cryptic-",
    "-backgroundbb",
    "-focalbb",

]:
    prefix = ""
    if "step" in bucket:
        step = int(bucket.split("-step")[-1]) + 1
    else:
        step = 0
    if "restint" in bucket:
        kind = bucket.split("-")[3]
    else:
        kind = None
    s3_handle = boto3.resource(
        's3',
        region_name="us-east-2",
        config=botocore.config.Config(
            signature_version=botocore.UNSIGNED,
        ),
    )
    bucket_handle = s3_handle.Bucket(bucket)

    for stint in tqdm([100]):
        try:
            series_profiles, = bucket_handle.objects.filter(
                Prefix=f'endeavor=16/{prefix}control-competitions-highestroot{suffix}/stage=4+what=collated/stint={stint}/',
            )
            import warnings
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
            control_fits_df = get_control_t_distns(bucket, prefix, suffix, 16, stint)
            df = pd.read_csv(
                f's3://{bucket}/{series_profiles.key}',
                compression='xz',
            )
            # df = df[df["Competition Series"] == 16005]
            # df = df.groupby([
            #     "genome variation"
            # ]).mean(numeric_only=True).reset_index()
            df["Stint"] = stint
            df["Series"] = df["Competition Series"]
            dfdigest = "{:x}".format( hash_pandas_object( df ).sum() )
            df = preprocess_competition_fitnesses(df, control_fits_df)
            assert "Series" in df.columns, df.columns

            df = df.copy()
            df["bucket"] = bucket
            df["variant"] = {"cryptic-": "skeleton", "": "wildtype"}[prefix]
            df["how"] = suffix.strip("-") if suffix else "none"
            dfs.append(df)
        except Exception as e:
            print(e)
            print(f"Skipping {bucket=}, {stint=}, {prefix=}")

print(len(dfs), "dataframes loaded")
df = pd.concat(dfs)


In [ ]:
plt.rcParams['figure.dpi'] = 600
plt.rcParams['savefig.dpi'] = 600


In [ ]:
palette = ["#65952f", "#79c2ac"]

with tp.teed(
    plt.subplots,
    1,
    2,
    figsize=(5, 2),
    sharey=True,
    gridspec_kw={"width_ratios": [3.5, 1]},
) as (fig, (ax1, ax2)):
    dfx["site"] = dfx["site"].astype(int)  # Ensure site is numeric for proper sorting

    # 1. Isolate and filter the sites (same as before)
    valid_sites = dfx[dfx["Prevalence"] < 0.25 * 0.75]["site"].unique()
    df_filtered = dfx[dfx["site"].isin(valid_sites)]

    m = MarkerStyle("d")
    m._transform.rotate_deg(90)
    # 2. Draw the scatterplot
    sns.scatterplot(
        data=df_filtered,
        ax=ax1,
        x="site",
        y="Prevalence",
        style="how",
        hue="how",
        s=30,      # Slightly larger dots look better on connected plots
        zorder=2,   # Forces the dots to be drawn ON TOP of the lines
        alpha=0.7,
        linewidth=0.6,
        markers=["d", m],
        legend=False,
        palette=palette,
    )

    sns.scatterplot(
        data=dfx,
        x="site",
        y="Prevalence",
        style="how",
        hue="how",
        s=10,      # Slightly larger dots look better on connected plots
        alpha=0.3,  # Make the background points more transparent
        zorder=-1,  # Forces the dots to be drawn BEHIND the lines
        legend=False,  # Avoid duplicate legends
        ax=ax1,
        palette=palette,
        markers=["d", m],
    )

    # 3. Find the min and max for each site to get our line endpoints
    ranges = df_filtered.groupby("site")["Prevalence"].agg(["min", "max"])

    # 4. Draw the connecting vertical lines
    ax1.vlines(
        x=ranges.index,
        ymin=ranges["min"],
        ymax=ranges["max"],
        color="k",
        linestyle=":", # Optional: makes the connection a dashed line
        linewidth=0.5,   # Optional: makes the line thinner
        alpha=0.9,      # Optional: softens the line color
        zorder=1        # Forces the lines to be drawn BEHIND the dots
    )

    # 5. Position the legend
    ax1.axhline(0.25, color="gray", linestyle="--", zorder=-1, alpha=0.5)  # Optional: adds a horizontal reference line at y=0.25

    how_categories = df[df["Root ID"] == 0]["how"].dropna().unique()
    swarm_markers = ["d", m]

    # 2. Loop through each category and plot it on ax2
    for i, category in enumerate(how_categories):
        subset = df[(df["Root ID"] == 0) & (df["how"] == category)]

        sns.swarmplot(
            data=subset,
            y="Prevalence",
            x="how",
            order=how_categories,  # Forces Seaborn to maintain the 2-column layout
            marker=swarm_markers[i],
            color=palette[i],      # Applies the exact color from your palette
            s=3,
            alpha=0.5,
            zorder=10,
            ax=ax2,
            legend=False           # Prevents duplicate legend entries
        )

    # 5. Position the legend
    ax2.axhline(0.25, color="gray", linestyle="--", zorder=-1, alpha=0.5)  # Optional: adds a horizontal reference line at y=0.25
    ax1.set_xlabel("Knockout Site")
    ax2.set_xlabel("Control")
    ax2.set_xticks([])
    ax1.set_ylabel("Fitness Result")

    sns.despine(ax=ax1)
    sns.despine(ax=ax2, left=True)
    # sns.move_legend(
    #     ax1, "lower center",
    #     bbox_to_anchor=(.5, 1), ncol=3, title=None, frameon=False,
    # )


    focal_sites = [
        36, 59, 63, 74, 167, 242, 349,
        1242, 1245, 1407, 1414, 1954, 2033, 2038, 2039
    ]

    # Get the current bottom limit of the y-axis so the lines start exactly at the bottom
    ymin, _ = ax1.get_ylim()

    focal_sites = [
        36, 59, 63, 74, 167, 242, 349,
        1242, 1245, 1407, 1414, 1954, 2033, 2038, 2039
    ]

    for i, site in enumerate(focal_sites):
        # Find the top of the point for this specific site
        # (Using max() in case a site has multiple points to ensure it starts at the very top)
        point_top = dfx[dfx["site"] == site]["Prevalence"].max()

        # Alternate the end height between 0.3, 0.35, and 0.4 based on the index
        y_target =[0.4, 0.35, 0.3][(i - 1) % 3]

        if site in [1954, 1242, 1245, 1407, 1414]:
            y_target = 0.3
        if site == 59:
            y_target = 0.3
        if site == 63:
            y_target = 0.35
        if site == 74:
            y_target = 0.4

        # Draw a vertical dashed black line from the point up to the staggered height
        ax1.plot(
            [site, site], [point_top, y_target + 0.03],
            color="black",
            linestyle=":",
            linewidth=0.5,
            alpha=0.5,
            zorder=0,
        )

        rotation = 60
        ha="left"
        va="bottom"
        if site in [1407, 1414]:
            if site == 1407:
                site_text = "    1407,1414"
            else:
                site_text = ""
            site = 1330
        elif site in [1242, 1245]:
            if site == 1242:
                site_text = "    1242,1245"
            else:
                site_text = ""
            site = 1242 - 77
        elif site == 1954:
            rotation = 90
            site_text = " " * 4 + str(site)
            ha="center"
            va="bottom"
        elif site == 2033:
            y_target = 0.4
            site_text = f" {site},"
        elif site == 2038:
            y_target = 0.35
            site_text = " " * 2 + str(site) + ","
        elif site == 2039:
            y_target = 0.3
            site_text = " " * 3 + str(site)
        elif site == 36:
            ha = "right"
            site_text = str(site)
        elif site == 74:
            va = "bottom"
            site_text = str(site)
            y_target += 0.03
        else:
            site_text = str(site)


        # Add the text label at the top of the line
        ax1.text(
            x=site,
            y=y_target,
            s=f"{site_text}",
            rotation=rotation,
            color="black",
            ha=ha,
            va=va,
            fontsize=6,
            zorder=3
        )

    # Prevent the manual lines from artificially pushing the bottom of the graph down
    ax1.set_ylim(bottom=ymin)

    ax2.text(
        x=0,
        y=0.35,
        s="Co-Evo\nBkgd.",
        rotation=90,
        color="black",
        ha="center",
        va="bottom",
        fontsize=8,
        zorder=3
    )

    ax2.text(
        x=1,
        y=0.35,
        s="Self\nBkgd.",
        rotation=90,
        color="black",
        ha="center",
        va="bottom",
        fontsize=8,
        zorder=3
    )

    ax2.yaxis.set_visible(False)

    fig.tight_layout()
